# Part A — Intermediate Python Programming

This notebook answers Problems 1–4. Run the cells from top to bottom.

## Problem 1. Functions and modular programming

In [ ]:
import math
from statistics import mean, median, pstdev

def calculate_statistics(values):
    valid = [float(x) for x in values if is_valid_number(x)]
    if not valid:
        return {"mean": None, "median": None, "standard_deviation": None}
    return {
        "mean": mean(valid),
        "median": median(valid),
        "standard_deviation": pstdev(valid)
    }

def min_max_normalize(values):
    valid = [float(x) for x in values if is_valid_number(x)]
    if not valid:
        return []
    low, high = min(valid), max(valid)
    if low == high:
        return [0.0 for _ in valid]
    return [(x - low) / (high - low) for x in valid]

def is_valid_number(value):
    try:
        return value is not None and math.isfinite(float(value))
    except (TypeError, ValueError):
        return False

def find_invalid_values(values):
    return [(index, value) for index, value in enumerate(values)
            if not is_valid_number(value)]

def summary_report(values):
    invalid = find_invalid_values(values)
    stats = calculate_statistics(values)
    return {
        "input_count": len(values),
        "valid_count": len(values) - len(invalid),
        "invalid_values": invalid,
        **stats
    }

values = [10, 20, 30, None, "bad", 40]
print("Summary report:", summary_report(values))
print("Min-max normalized:", min_max_normalize([10, 20, 30, 40]))

## Problem 2. Lists, dictionaries and data processing

In [ ]:
students = [
    {"name": "Anu", "attendance": 92, "assignment_marks": 85, "exam_marks": 88},
    {"name": "Basil", "attendance": 80, "assignment_marks": 76, "exam_marks": 79},
    {"name": "Diya", "attendance": 96, "assignment_marks": 94, "exam_marks": 91},
    {"name": "Farhan", "attendance": 85, "assignment_marks": 82, "exam_marks": 84},
    {"name": "Meera", "attendance": 89, "assignment_marks": 90, "exam_marks": 95},
]

for student in students:
    student["average_mark"] = (student["assignment_marks"] + student["exam_marks"]) / 2

class_average = sum(s["average_mark"] for s in students) / len(students)
above_average = [s for s in students if s["average_mark"] > class_average]
ranked_students = sorted(students, key=lambda s: s["average_mark"], reverse=True)

print(f"Class average: {class_average:.2f}")
print("Students above class average:", [s["name"] for s in above_average])
print("Students sorted by marks:")
for student in ranked_students:
    print(student["name"], f'{student["average_mark"]:.2f}')
print("Top three:", [(s["name"], s["average_mark"]) for s in ranked_students[:3]])

## Problem 3. File handling and exception handling

In [ ]:
import csv
from pathlib import Path

input_file = Path("student_records_input.csv")
output_file = Path("student_records_valid.csv")

with input_file.open("w", newline="", encoding="utf-8") as file:
    writer = csv.DictWriter(file, fieldnames=["name", "attendance", "marks"])
    writer.writeheader()
    writer.writerows([
        {"name": "Anu", "attendance": "92", "marks": "88"},
        {"name": "Basil", "attendance": "81", "marks": "76"},
        {"name": "", "attendance": "75", "marks": "80"},
        {"name": "Diya", "attendance": "105", "marks": "91"},
        {"name": "Farhan", "attendance": "absent", "marks": "79"},
    ])

valid_records = []
invalid_rows = []

try:
    with input_file.open("r", newline="", encoding="utf-8") as file:
        for row_number, row in enumerate(csv.DictReader(file), start=2):
            try:
                name = row["name"].strip()
                attendance = float(row["attendance"])
                marks = float(row["marks"])
                if not name:
                    raise ValueError("name is missing")
                if not 0 <= attendance <= 100:
                    raise ValueError("attendance must be between 0 and 100")
                if not 0 <= marks <= 100:
                    raise ValueError("marks must be between 0 and 100")
                valid_records.append({"name": name, "attendance": attendance, "marks": marks})
            except (ValueError, TypeError, KeyError) as error:
                invalid_rows.append((row_number, str(error)))

    with output_file.open("w", newline="", encoding="utf-8") as file:
        writer = csv.DictWriter(file, fieldnames=["name", "attendance", "marks"])
        writer.writeheader()
        writer.writerows(valid_records)

    print("Valid records written:", len(valid_records))
    print("Invalid rows:", invalid_rows)
    print("Output file:", output_file.resolve())
except OSError as error:
    print("File operation failed:", error)

## Problem 4. Object-oriented programming

In [ ]:
import pandas as pd

class MLDataset:
    def __init__(self, file_path, separator=","):
        self.file_path = file_path
        self.separator = separator
        self.data = None

    def load_dataset(self):
        self.data = pd.read_csv(self.file_path, sep=self.separator)
        return self.data

    def display_dimensions(self):
        if self.data is None:
            raise ValueError("Load the dataset before checking its dimensions.")
        print(f"Rows: {self.data.shape[0]}, Columns: {self.data.shape[1]}")

    def identify_missing_values(self):
        if self.data is None:
            raise ValueError("Load the dataset first.")
        return self.data.isna().sum()

    def select_columns(self, columns):
        if self.data is None:
            raise ValueError("Load the dataset first.")
        missing = [column for column in columns if column not in self.data.columns]
        if missing:
            raise KeyError(f"Unknown columns: {missing}")
        return self.data[columns]

sample_file = Path("sample_dataset.csv")
pd.DataFrame({"name": ["Asha", "Ravi"], "marks": [88, 75]}).to_csv(sample_file, index=False)
dataset = MLDataset(sample_file)
dataset.load_dataset()
dataset.display_dimensions()
print("Missing values:")
print(dataset.identify_missing_values())
print("Selected columns:")
print(dataset.select_columns(["name", "marks"]))